# Import necessary libraries

In [1]:
import numpy as np
import pandas as pd
from mlxtend.classifier import StackingClassifier
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from tabulate import tabulate
from joblib import dump, load

# Load dataset

In [2]:
df = pd.read_csv('../data/training_data.csv')
df.head()

,Diabetes,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,0.0,1.0,1.0,15.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,5.0,10.0,20.0,0.0,0.0,11.0,4.0,5.0
1,1.0,1.0,0.0,1.0,28.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,4.0,3.0
2,1.0,1.0,1.0,1.0,33.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,10.0,0.0,0.0,0.0,9.0,4.0,7.0
3,1.0,0.0,1.0,1.0,29.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,5.0,0.0,30.0,1.0,1.0,12.0,3.0,4.0
4,0.0,0.0,0.0,1.0,24.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,1.0,1.0,13.0,5.0,6.0


# Define some constants

In [3]:
LABEL_COL = 'Diabetes'
CATEGORICAL_COLS = ['GenHlth', 'Age', 'Education', 'Income']
NUMERICAL_COLS = ['BMI', 'MentHlth', 'PhysHlth']

NUM_FOLDS = 5

# Meta model for stacking ensemble
META_MODEL = LogisticRegression(solver='liblinear',
                                class_weight='balanced',
                                random_state=42)

# Data preprocessing

In [4]:
# Separate features and labels
X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL]

# Reuse scaler and encoder from previous script
scaler = load('../models/scaler.bin')
encoder = load('../models/label_encoders.bin')

# Preprocess numerical features
X[NUMERICAL_COLS] = scaler.transform(X[NUMERICAL_COLS])

# Preprocess categorical features
for col in CATEGORICAL_COLS:
    X[col] = encoder[col].transform(X[col])
    
# Double check
X.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,-2.102124,1.0,0.0,0.0,0.0,0.0,1.0,0.0,4,0.643339,1.712247,0.0,0.0,10,3,4
1,1.0,0.0,1.0,-0.183495,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1,-0.549621,-0.522861,0.0,0.0,10,3,2
2,1.0,1.0,1.0,0.554440,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1,0.643339,-0.522861,0.0,0.0,8,3,6
3,0.0,1.0,1.0,-0.035908,0.0,1.0,1.0,1.0,0.0,1.0,0.0,4,-0.549621,2.829801,1.0,1.0,11,2,3
4,0.0,0.0,1.0,-0.773842,1.0,0.0,0.0,0.0,0.0,1.0,0.0,2,-0.549621,-0.522861,1.0,1.0,12,4,5


# Tuning hyperparameters for base models

## `LogisticRegression`

In [5]:
# Tune a base LogisticRegression with GridSearchCV (independent from meta-model)

# Define parameter grid for LogisticRegression
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [100, 200, 500]
}

lr_base = LogisticRegression(random_state=42)
gs_lr = GridSearchCV(lr_base, param_grid_lr, cv=NUM_FOLDS, scoring='f1', n_jobs=-1, verbose=2)
gs_lr.fit(X, y)

print("Best params for LogisticRegression:", gs_lr.best_params_)
print("Best CV f1 score for LogisticRegression:", gs_lr.best_score_)

# Update lr_model with the best estimator from GridSearchCV
lr_model = gs_lr.best_estimator_

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best params for LogisticRegression: {'C': 0.01, 'class_weight': 'balanced', 'max_iter': 100, 'solver': 'liblinear'}
Best CV f1 score for LogisticRegression: 0.484107444514161


## `SGDClassifier`

In [6]:
# Tune SGDClassifier with GridSearchCV

# Define parameter grid for SGDClassifier
param_grid_sgd = {
    'loss': ['hinge', 'log_loss', 'modified_huber', 'perceptron'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'class_weight': [None, 'balanced'],
    'max_iter': [100, 200, 500, 1000]
}

sgd_base = SGDClassifier(random_state=42)
gs_sgd = GridSearchCV(sgd_base, param_grid_sgd, cv=NUM_FOLDS, scoring='f1', n_jobs=-1, verbose=2)
gs_sgd.fit(X, y)

print("Best params for SGDClassifier:", gs_sgd.best_params_)
print("Best CV f1 score for SGDClassifier:", gs_sgd.best_score_)

# Update sgd_model with the best estimator from GridSearchCV
sgd_model = gs_sgd.best_estimator_

Fitting 5 folds for each of 128 candidates, totalling 640 fits
Best params for SGDClassifier: {'alpha': 0.001, 'class_weight': 'balanced', 'loss': 'log_loss', 'max_iter': 100}
Best CV f1 score for SGDClassifier: 0.48340108476539


## `DecisionTreeClassifier`

In [7]:
# Tune DecisionTreeClassifier with GridSearchCV

# Define parameter grid for DecisionTreeClassifier
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 10],
    'class_weight': [None, 'balanced']
}

dt_base = DecisionTreeClassifier(random_state=42)
gs = GridSearchCV(dt_base, param_grid, cv=NUM_FOLDS, scoring='f1', n_jobs=-1, verbose=2)
gs.fit(X, y)

print("Best params:", gs.best_params_)
print("Best CV f1 score:", gs.best_score_)

# Update dt_model with the best estimator from GridSearchCV
dt_model = gs.best_estimator_

Fitting 5 folds for each of 192 candidates, totalling 960 fits
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best CV f1 score: 0.4746178634378582


## `GaussianNB`

In [8]:
# Tune GaussianNB with GridSearchCV

# Define parameter grid for GaussianNB
param_grid_nb = {
    'var_smoothing': [1e-09, 1e-08, 1e-07, 1e-06]
}

gnb_base = GaussianNB()
gs_gnb = GridSearchCV(gnb_base, param_grid_nb, cv=NUM_FOLDS, scoring='f1', n_jobs=-1, verbose=2)
gs_gnb.fit(X, y)

print("Best params for GaussianNB:", gs_gnb.best_params_)
print("Best CV f1 score for GaussianNB:", gs_gnb.best_score_)

# Update gnb_model with the best estimator from GridSearchCV
gnb_model = gs_gnb.best_estimator_

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best params for GaussianNB: {'var_smoothing': 1e-06}
Best CV f1 score for GaussianNB: 0.4483422794536085


## `LinearSVC`

In [9]:
# Tune LinearSVC with GridSearchCV

# Define parameter grid for LinearSVC
param_grid_svc = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
    'max_iter': [100, 200, 500]
}

svc_base = LinearSVC(random_state=42)
gs_svc = GridSearchCV(svc_base, param_grid_svc, cv=NUM_FOLDS, scoring='f1', n_jobs=-1, verbose=2)
gs_svc.fit(X, y)

print("Best params for LinearSVC:", gs_svc.best_params_)
print("Best CV f1 score for LinearSVC:", gs_svc.best_score_)

# Update svc_model with the best estimator from GridSearchCV
svc_model = gs_svc.best_estimator_

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params for LinearSVC: {'C': 1, 'class_weight': 'balanced', 'max_iter': 100}
Best CV f1 score for LinearSVC: 0.48285307719990406


# K-Fold Cross Validation

## Cross-validation training

In [11]:
# Base models for stacking ensemble
base_models = [lr_model, sgd_model, dt_model, gnb_model, svc_model]

# K-fold Cross Validation Training
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

fold = 1 # Fold counter
classification_reports = [] # To store classification reports for each fold

for train_index, val_index in kf.split(X):
    print(f"Training fold {fold}...")
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Define Stacking Classifier
    stack_clf = StackingClassifier(classifiers=base_models, meta_classifier=META_MODEL)
    
    # Train the stacking classifier
    stack_clf.fit(X_train, y_train)
    
    # Evaluate on validation set
    y_pred = stack_clf.predict(X_val)
    cr = classification_report(y_val, y_pred, output_dict=True)
    cm = confusion_matrix(y_val, y_pred)
    print(f"Fold {fold} results:")
    print(cr)
    print("Confusion Matrix:")
    print(cm)
    
    # Store report
    classification_reports.append(cr)
    
    print("\n")
    fold += 1

Training fold 1...
Fold 1 results:
{'0.0': {'precision': 0.9227126701454011, 'recall': 0.6820165806522456, 'f1-score': 0.7843134159912238, 'support': 72856.0}, '1.0': {'precision': 0.35905381103887124, 'recall': 0.7571761960326722, 'f1-score': 0.48711644928216197, 'support': 17140.0}, 'accuracy': 0.6963309480421352, 'macro avg': {'precision': 0.6408832405921362, 'recall': 0.7195963883424589, 'f1-score': 0.6357149326366929, 'support': 89996.0}, 'weighted avg': {'precision': 0.815362200734695, 'recall': 0.6963309480421352, 'f1-score': 0.7277113891301041, 'support': 89996.0}}
Confusion Matrix:
[[49689 23167]
 [ 4162 12978]]


Training fold 2...
Fold 2 results:
{'0.0': {'precision': 0.9203200114802325, 'recall': 0.7012273460350436, 'f1-score': 0.7959725088043191, 'support': 73166.0}, '1.0': {'precision': 0.3617145526746087, 'recall': 0.7360665478312537, 'f1-score': 0.4850620619444771, 'support': 16830.0}, 'accuracy': 0.7077425663362816, 'macro avg': {'precision': 0.6410172820774206, 'recal

## Cross-validation average results

In [14]:
avg_report = {}

# Extract average metrics across folds
for key in classification_reports[0].keys():
    if key in ['0.0', '1.0', 'macro avg', 'weighted avg']: # Average the metric sections
        avg_report[key] = {}
        for metric in classification_reports[0][key].keys(): # Iterate through metrics ('precision', 'recall', etc.)
            avg_report[key][metric] = np.mean([report[key][metric] for report in classification_reports])
    elif key == 'accuracy':
        avg_report[key] = np.mean([report[key] for report in classification_reports])

# Format and print the average classification report
headers = ["precision", "recall", "f1-score", "support"]
table = []
for label in ['0.0', '1.0', 'macro avg', 'weighted avg']:
    if label in avg_report:
        row = [
            label,
            f"{avg_report[label]['precision']:.4f}",
            f"{avg_report[label]['recall']:.4f}",
            f"{avg_report[label]['f1-score']:.4f}",
            f"{int(avg_report[label]['support']):d}"
        ]
        table.append(row)

print(tabulate(table, headers=headers, floatfmt=".4f", numalign="right"))
print(f"\nAverage Accuracy: {avg_report['accuracy']:.4f}")

                precision    recall    f1-score    support
------------  -----------  --------  ----------  ---------
0.0                0.9176    0.7054      0.7975      73009
1.0                0.3650    0.7274      0.4860      16986
macro avg          0.6413    0.7164      0.6417      89995
weighted avg       0.8133    0.7095      0.7387      89995

Average Accuracy: 0.7095


# Train final model on entire dataset

In [15]:
# Train Final Model on Full Dataset
final_model = StackingClassifier(classifiers=base_models, meta_classifier=META_MODEL)
final_model.fit(X, y)

# Save the final stacking ensemble model
dump(final_model, '../models/stacking_ensemble_model.bin')

['../models/stacking_ensemble_model.bin']